In [135]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

import umap
from sklearn.decomposition import KernelPCA
from sklearn.decomposition import PCA
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn.metrics import accuracy_score

from imblearn.over_sampling import SMOTE


In [136]:
#데이터셋 로드
train_df = pd.read_csv('./채무불이행/train.csv').drop(columns = ['UID'])
test_df = pd.read_csv('./채무불이행/test.csv').drop(columns = ['UID'])

In [137]:
#데이터셋 분리
X_df=train_df.drop(columns='채무 불이행 여부')
y_df=train_df['채무 불이행 여부']

In [138]:
# 범주형 데이터
cat_col = [
    '주거 형태',
    '현재 직장 근속 연수',
    '대출 목적',
    '대출 상환 기간'
]

In [139]:
# Mapping 0.0 - One-Hot
cat_col = [
    '주거 형태',
    '현재 직장 근속 연수',
    '대출 목적',
    '대출 상환 기간'
]
encoder_ohe=OneHotEncoder(sparse_output=False)
encoded=encoder_ohe.fit_transform(X_df[cat_col])
encoded_df = pd.DataFrame(encoded, columns=encoder_ohe.get_feature_names_out(cat_col))
X_ohe = pd.concat([X_df.drop(columns=cat_col).reset_index(drop=True), encoded_df], axis=1)

In [140]:
# Mapping 0.1 - 대출 상환 기간 0-1
cat_col_no_term = [
    '주거 형태',
    '현재 직장 근속 연수',
    '대출 목적'
]
encoder_term=OneHotEncoder(sparse_output=False)
encoded=encoder_term.fit_transform(X_df[cat_col_no_term])
encoded_df = pd.DataFrame(encoded, columns=encoder_term.get_feature_names_out(cat_col_no_term))
X_term = pd.concat([X_df.drop(columns=cat_col_no_term).reset_index(drop=True), encoded_df], axis=1)

X_term['대출 상환 기간']=np.where(X_term['대출 상환 기간']== '단기 상환', 1, 0)

In [141]:
# Mapping 1 - 주거 형태 labeling
house_mapping = {'월세' : 0, '자가' : 1 , '주택 담보 대출 (거주 중)' : 2, '주택 담보 대출 (비거주 중)' : 3}

cat_col_no_house = [
    '현재 직장 근속 연수',
    '대출 목적'
]
encoder_house=OneHotEncoder(sparse_output=False)
encoded=encoder_house.fit_transform(X_df[cat_col_no_house])
encoded_df = pd.DataFrame(encoded, columns=encoder_house.get_feature_names_out(cat_col_no_house))
X_house = pd.concat([X_df.drop(columns=cat_col_no_house).reset_index(drop=True), encoded_df], axis=1)
X_house['대출 상환 기간']=np.where(X_house['대출 상환 기간']== '단기 상환', 1, 0)


X_house['주거 형태']=X_house['주거 형태'].map(house_mapping)

In [142]:
# Mapping 2 - 근속 연수 실수화
cat_col_no_year = [
    '주거 형태',
    '대출 목적'
]
encoder_year=OneHotEncoder(sparse_output=False)
encoded=encoder_year.fit_transform(X_df[cat_col_no_year])
encoded_df = pd.DataFrame(encoded, columns=encoder_year.get_feature_names_out(cat_col_no_year))
X_year = pd.concat([X_df.drop(columns=cat_col_no_year).reset_index(drop=True), encoded_df], axis=1)
X_year['대출 상환 기간']=np.where(X_year['대출 상환 기간']== '단기 상환', 1, 0)

X_year['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].str.replace('1년 미만', '0').str.replace('10년 이상', '10').str.replace('년', '').astype(int)

In [143]:
# Mapping 3 - 대출 목적 1개 (arbitrary)
cat_col_no_purpose = [
    '주거 형태',
    '현재 직장 근속 연수'
]


encoder_purpose=OneHotEncoder(sparse_output=False)
encoded=encoder_purpose.fit_transform(X_df[cat_col_no_purpose])
encoded_df = pd.DataFrame(encoded, columns=encoder_purpose.get_feature_names_out(cat_col_no_purpose))
X_purpose = pd.concat([X_df.drop(columns=cat_col_no_purpose).reset_index(drop=True), encoded_df], axis=1)
X_purpose['대출 상환 기간']=np.where(X_purpose['대출 상환 기간']== '단기 상환', 1, 0)

X_purpose['대출 목적']=np.where(X_purpose['대출 목적'] == '부채 통합', 1, 0)

In [144]:
# Mapping 2+3
cat_col_only_house=['주거 형태']

encoder_year=OneHotEncoder(sparse_output=False)
encoded=encoder_year.fit_transform(X_df[cat_col_only_house])
encoded_df = pd.DataFrame(encoded, columns=encoder_year.get_feature_names_out(cat_col_only_house))
X_year_purpose = pd.concat([X_df.drop(columns=cat_col_only_house).reset_index(drop=True), encoded_df], axis=1)
X_year_purpose['대출 상환 기간']=np.where(X_year_purpose['대출 상환 기간']== '단기 상환', 1, 0)

X_year_purpose['현재 직장 근속 연수'] = X_year_purpose['현재 직장 근속 연수'].str.replace('1년 미만', '0').str.replace('10년 이상', '10').str.replace('년', '').astype(int)
X_year_purpose['대출 목적']=np.where(X_year_purpose['대출 목적'] == '부채 통합', 1, 0)

In [145]:
def Scale(data, scaler, fit=False): #scale된 DataFrame 반환
    original_columns = data.columns
    if fit==True: scaler.fit(data)
    scaled_data = scaler.transform(data)
    scaled_df = pd.DataFrame(scaled_data, columns = original_columns)
    return scaled_df

In [165]:
def XGBmodeling(df,verbose=True,model=None, smote=None, detail_score=False, eval=2, fit_all=False): #df로 xgb 학습, 성능 출력, 학습된 xgb 반환


    X_train, X_val, y_train, y_val = train_test_split(
        df, 
        train_df['채무 불이행 여부'], 
        test_size=0.2, 
        random_state=42
    )

    if model is None:
        model = xgb.XGBClassifier(
        n_estimators = 150,
        max_depth = 5,
        learning_rate = 0.15,
        random_state=42,
        eval_metric="auc",
        )
    
    
    if smote is not None:
        X_train, y_train = smote.fit_resample(X_train, y_train)

    if eval==0:
        eval_set=None
    elif eval==1:
        eval_set=[(X_val,y_val)]
    elif eval==2:
        eval_set=[(X_train,y_train),(X_val,y_val)]
        
    if fit_all:
        X_fit=df
        y_fit=train_df['채무 불이행 여부']
    else:
        X_fit=X_train
        y_fit=y_train
        
    model.fit(
        X_fit, y_fit,
        eval_set = eval_set,
        verbose=verbose
        )
    
    if detail_score:
        print("train 성능: %.4f"%model.score(X_train,y_train))
        print("test 성능: %.4f"%model.score(X_val,y_val))
        
    print("모델 성능: %.4f"%model.score(df,train_df['채무 불이행 여부']))
    
    return model

In [147]:
mmscaler=MinMaxScaler()
stdscaler=StandardScaler()
robscaler=RobustScaler()
X_ohe_mms=Scale(X_ohe,mmscaler,fit=True)
X_ohe_std=Scale(X_ohe,mmscaler,fit=True)
X_ohe_rob=Scale(X_ohe,mmscaler,fit=True)

In [148]:
# Scaling -> 차이 X

XGBmodeling(X_ohe_mms, verbose=False)
XGBmodeling(X_ohe_std, verbose=False)
XGBmodeling(X_ohe_rob, verbose=False)
XGBmodeling(X_ohe, verbose=False)

모델 성능: 0.8328
모델 성능: 0.8328
모델 성능: 0.8328
모델 성능: 0.8328


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [149]:
# 대출 기간 - binary -> one-hot vs label 차이 없음
XGBmodeling(X_ohe, verbose=False)
XGBmodeling(X_term, verbose=False)

모델 성능: 0.8328
모델 성능: 0.8328


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [150]:
# year 10년 이상 숫자 가중 - 차이 X (XGB가 Tree 기반이라 양 끝단은 값 바꿔도 의미 없는듯 - year=<9 이런식으로 할거니까)

X_year_11=X_year.copy()
X_year_12=X_year.copy()
X_year_13=X_year.copy()
X_year_14=X_year.copy()
X_year_15=X_year.copy()
X_year_20=X_year.copy()

X_year_11['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,11)
X_year_12['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,12)
X_year_13['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,13)
X_year_14['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,14)
X_year_15['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,15)
X_year_20['현재 직장 근속 연수'] = X_year['현재 직장 근속 연수'].replace(10,20)

XGBmodeling(X_year, verbose=False)
XGBmodeling(X_year_11, verbose=False)
XGBmodeling(X_year_12, verbose=False)
XGBmodeling(X_year_13, verbose=False)
XGBmodeling(X_year_14, verbose=False)
XGBmodeling(X_year_15, verbose=False)
XGBmodeling(X_year_20, verbose=False)

모델 성능: 0.8269
모델 성능: 0.8269
모델 성능: 0.8269
모델 성능: 0.8269
모델 성능: 0.8269
모델 성능: 0.8269
모델 성능: 0.8269


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [151]:
# 한 범주씩 줄임 - 약간 변동

XGBmodeling(X_ohe, verbose=False) #1
XGBmodeling(X_year, verbose=False) #4
XGBmodeling(X_house, verbose=False) #3
XGBmodeling(X_purpose, verbose=False) #2

모델 성능: 0.8328
모델 성능: 0.8269
모델 성능: 0.8289
모델 성능: 0.8317


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [155]:
# XGBoost 학습 방법 변경 (X_train,y_train -> df,y_df) - 다른 경향 (1432 -> 3241)

for df in [X_ohe,X_year,X_house, X_purpose]: #3 #2 #4 #1
        XGBmodeling(df,verbose=False,fit_all=True)

모델 성능: 0.8470
모델 성능: 0.8479
모델 성능: 0.8441
모델 성능: 0.8491


In [170]:
# 모델 파라미터 차이 - 약간 다른 경향 (1432 -> 2431)

model_2 = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,  # 로지스틱 회귀는 보통 깊이를 작게 설정
    learning_rate=0.1,
    objective="binary:logistic",  # 로지스틱 회귀 설정
    eval_metric="auc",  # 손실 함수
    random_state=42
)

for df in [X_ohe,X_year,X_house, X_purpose]: #2 #4 #3 #1
    XGBmodeling(df, verbose=False, model=model_2)

모델 성능: 0.7679
모델 성능: 0.7656
모델 성능: 0.7670
모델 성능: 0.7688


In [166]:
#SMOTE 적용 - 줄어든거같은데용? 엥쓰? -eval set 때문인가...
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
for df in [X_ohe,X_year,X_purpose,X_year_purpose]:
    XGBmodeling(df, verbose=False, detail_score=True)
    
print("\n")
    
for df in [X_ohe,X_year,X_purpose,X_year_purpose]:
    XGBmodeling(df, verbose=False, smote=smote, detail_score=True)

train 성능: 0.8648
test 성능: 0.7050
모델 성능: 0.8328
train 성능: 0.8592
test 성능: 0.6975
모델 성능: 0.8269
train 성능: 0.8642
test 성능: 0.7015
모델 성능: 0.8317
train 성능: 0.8635
test 성능: 0.6935
모델 성능: 0.8295


train 성능: 0.8894
test 성능: 0.7050
모델 성능: 0.8248
train 성능: 0.8743
test 성능: 0.6995
모델 성능: 0.8128
train 성능: 0.8890
test 성능: 0.7015
모델 성능: 0.8262
train 성능: 0.8829
test 성능: 0.6955
모델 성능: 0.8235


In [ ]:
#eval_set 다르게 - 차이 X - 조기종료 X여서??

for df in [X_ohe,X_year,X_purpose,X_year_purpose]:
    XGBmodeling(df, verbose=False, smote=smote, detail_score=True, eval=0)

print("\n")
 
for df in [X_ohe,X_year,X_purpose,X_year_purpose]:
    XGBmodeling(df, verbose=False, smote=smote, detail_score=True, eval=1)

train 성능: 0.8894
test 성능: 0.7050
모델 성능: 0.8248
train 성능: 0.8743
test 성능: 0.6995
모델 성능: 0.8128
train 성능: 0.8890
test 성능: 0.7015
모델 성능: 0.8262
train 성능: 0.8829
test 성능: 0.6955
모델 성능: 0.8235


train 성능: 0.8894
test 성능: 0.7050
모델 성능: 0.8248
train 성능: 0.8743
test 성능: 0.6995
모델 성능: 0.8128
train 성능: 0.8890
test 성능: 0.7015
모델 성능: 0.8262
train 성능: 0.8829
test 성능: 0.6955
모델 성능: 0.8235


In [ ]:
# 모델 파라미터 바꿔서 - 그래도 안좋음

for df in [X_ohe,X_year,X_house, X_purpose]:
    XGBmodeling(df, verbose=False,smote=smote, model=model_2)

모델 성능: 0.7632
모델 성능: 0.7576
모델 성능: 0.7576
모델 성능: 0.7638


In [ ]:
'''성능 측정 방법이 별로인듯,,, 다음에 F-1 score 써서 해보겟음'''